### Setup

In [0]:
# ============================================================
#  03_GOLD_AGGREGATION — TOURGO DATA PIPELINE
#  Mục đích: Silver Tables → Gold KPI Tables (3 bảng đầu)
#  Gold Table 1: gold_revenue_daily
#  Gold Table 2: gold_provider_performance
#  Gold Table 3: gold_tour_performance
# ============================================================

from pyspark.sql.functions import (
    col, sum as _sum, count, avg, round as _round,
    countDistinct, to_date, date_format,
    when, lit
)
from pyspark.sql.types import DoubleType

spark.sql("USE tourgo")
print("   Database: tourgo")
print("   Đọc từ: silver_bookings, silver_revenues, silver_tours, silver_users, silver_reviews")
print("   Ghi ra: gold_revenue_daily, gold_provider_performance, gold_tour_performance")

   Database: tourgo
   Đọc từ: silver_bookings, silver_revenues, silver_tours, silver_users, silver_reviews
   Ghi ra: gold_revenue_daily, gold_provider_performance, gold_tour_performance


### Cell 2 — Đọc Silver tables

In [0]:
# ── Đọc toàn bộ Silver tables cần dùng ─────────────────────
df_bookings = spark.read.table("silver_bookings")
df_revenues  = spark.read.table("silver_revenues")
df_tours     = spark.read.table("silver_tours")
df_users     = spark.read.table("silver_users")
df_payments  = spark.read.table("silver_payments")
df_reviews   = spark.read.table("silver_reviews")

# Verify nhanh
print("  Silver tables loaded:")
print(f"   bookings : {df_bookings.count():,} records")
print(f"   revenues : {df_revenues.count():,} records")
print(f"   tours    : {df_tours.count():,} records")
print(f"   users    : {df_users.count():,} records")
print(f"   payments : {df_payments.count():,} records")
print(f"   reviews  : {df_reviews.count():,} records")

  Silver tables loaded:
   bookings : 11 records
   revenues : 4 records
   tours    : 23 records
   users    : 18 records
   payments : 10 records
   reviews  : 1 records


### Cell 3 — Gold Table 1: Revenue Daily

In [0]:
# ── GOLD TABLE 1: Revenue by Day ───────────────────────────
# Logic: revenues JOIN bookings để lấy ngày
# Dùng revenues.total_amount vì chỉ tồn tại khi payment SUCCESS thật

print("\n  Đang tạo gold_revenue_daily...")

# revenues có payment_id → join với bookings.id để lấy booking_date
df_rev_booking = df_revenues.alias("rev").join(
    df_bookings.select(
        col("id").alias("booking_id"),
        col("tour_id"),
        to_date(col("created_at")).alias("booking_date")
    ).alias("bk"),
    col("rev.payment_id") == col("bk.booking_id"),
    "left"
)

df_gold_revenue_daily = df_rev_booking \
    .groupBy("booking_date") \
    .agg(
        _sum("total_amount").alias("total_revenue"),
        _sum("creator_share").alias("provider_revenue"),
        _sum("admin_share").alias("platform_fee"),
        count("rev.id").alias("num_transactions")
    ) \
    .withColumn("month", date_format(col("booking_date"), "yyyy-MM")) \
    .orderBy("booking_date")

df_gold_revenue_daily.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_revenue_daily")

result_count = spark.read.table("gold_revenue_daily").count()
print(f" gold_revenue_daily: {result_count} ngày có doanh thu")
spark.read.table("gold_revenue_daily").show(truncate=False)


  Đang tạo gold_revenue_daily...
 gold_revenue_daily: 2 ngày có doanh thu
+------------+-------------+----------------+------------+----------------+-------+
|booking_date|total_revenue|provider_revenue|platform_fee|num_transactions|month  |
+------------+-------------+----------------+------------+----------------+-------+
|NULL        |4000000.0    |3600000.0       |400000.0    |1               |NULL   |
|2026-05-24  |1140000.0    |1026000.0       |114000.0    |3               |2026-05|
+------------+-------------+----------------+------------+----------------+-------+



### Cell 4 — Gold Table 2: Provider Performance

In [0]:
# ── GOLD TABLE 2: Provider Performance ─────────────────────
# Logic: confirmed bookings → join tours (lấy creator_id)
#        → join revenues (lấy creator_share)
#        → join reviews (lấy avg rating)

print("\n  Đang tạo gold_provider_performance...")

# Bước 1: bookings confirmed → join tours để có creator_id
df_confirmed = df_bookings \
    .filter(col("status") == "confirmed") \
    .join(
        df_tours.select(
            col("id").alias("tour_ref_id"),
            col("creator_id"),
            col("title").alias("tour_title")
        ),
        col("tour_id") == col("tour_ref_id"),
        "left"
    )

# Bước 2: join revenues để có creator_share
df_with_rev = df_confirmed.join(
    df_revenues.select(
        col("payment_id"),
        col("creator_share"),
        col("total_amount").alias("rev_total")
    ),
    df_confirmed.id == col("payment_id"),
    "left"
)

# Bước 3: aggregate theo creator_id
df_gold_provider = df_with_rev \
    .groupBy("creator_id") \
    .agg(
        _sum("creator_share").alias("total_provider_revenue"),
        _sum("rev_total").alias("total_gmv"),
        count("id").alias("total_confirmed_bookings"),
        countDistinct("tour_id").alias("active_tours_count")
    )

# Bước 4: join avg rating từ reviews → tours → creator
df_provider_reviews = df_reviews \
    .join(
        df_tours.select(
            col("id").alias("t_id"),
            col("creator_id").alias("t_creator_id")
        ),
        col("tour_id") == col("t_id"),
        "left"
    ) \
    .groupBy("t_creator_id") \
    .agg(_round(avg("rating"), 2).alias("avg_rating")) \
    .withColumnRenamed("t_creator_id", "creator_id_r")

df_gold_provider = df_gold_provider \
    .join(
        df_provider_reviews,
        df_gold_provider.creator_id == df_provider_reviews.creator_id_r,
        "left"
    ) \
    .drop("creator_id_r") \
    .orderBy(col("total_provider_revenue").desc())

df_gold_provider.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_provider_performance")

result_count = spark.read.table("gold_provider_performance").count()
print(f" gold_provider_performance: {result_count} providers")
spark.read.table("gold_provider_performance").show(truncate=False)


  Đang tạo gold_provider_performance...
 gold_provider_performance: 2 providers
+----------+----------------------+---------+------------------------+------------------+----------+
|creator_id|total_provider_revenue|total_gmv|total_confirmed_bookings|active_tours_count|avg_rating|
+----------+----------------------+---------+------------------------+------------------+----------+
|11        |900000.0              |1000000.0|5                       |3                 |5.0       |
|17        |NULL                  |NULL     |2                       |1                 |NULL      |
+----------+----------------------+---------+------------------------+------------------+----------+



### Cell 5 — Gold Table 3: Tour Performance

In [0]:
# ── GOLD TABLE 3: Tour Performance ─────────────────────────
# Logic: confirmed bookings → join revenues → group by tour_id
#        → join reviews → join tours (lấy title, price, category)

print("\n  Đang tạo gold_tour_performance...")

# Bước 1: confirmed bookings → join revenues
df_tour_rev = df_bookings \
    .filter(col("status") == "confirmed") \
    .join(
        df_revenues.select(
            col("payment_id"),
            col("total_amount").alias("rev_total")
        ),
        df_bookings.id == col("payment_id"),
        "left"
    ) \
    .groupBy("tour_id") \
    .agg(
        count("id").alias("total_bookings"),
        _sum("rev_total").alias("total_revenue"),
        _sum("number_of_people").alias("total_travelers")
    )

# Bước 2: avg rating từ reviews
df_tour_ratings = df_reviews \
    .groupBy("tour_id") \
    .agg(
        _round(avg("rating"), 2).alias("avg_rating"),
        count("id").alias("review_count")
    )

# Bước 3: join tour info + ratings
df_gold_tours = df_tour_rev \
    .join(
        df_tours.select(
            col("id").alias("t_id"),
            col("title"),
            col("price"),
            col("category_names")
        ),
        col("tour_id") == col("t_id"),
        "left"
    ) \
    .drop("t_id") \
    .join(
        df_tour_ratings,
        "tour_id",
        "left"
    ) \
    .orderBy(col("total_revenue").desc())

df_gold_tours.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_tour_performance")

result_count = spark.read.table("gold_tour_performance").count()
print(f" gold_tour_performance: {result_count} tours")
spark.read.table("gold_tour_performance").show(truncate=False)


  Đang tạo gold_tour_performance...
 gold_tour_performance: 4 tours
+-------+--------------+-------------+---------------+---------------------------------+---------+-------------------+----------+------------+
|tour_id|total_bookings|total_revenue|total_travelers|title                            |price    |category_names     |avg_rating|review_count|
+-------+--------------+-------------+---------------+---------------------------------+---------+-------------------+----------+------------+
|12     |2             |1000000.0    |2              |Khám phá Hội An - Tour 3N2Đ      |4000000.0|NULL               |5.0       |1           |
|11     |2             |NULL         |5              |Sun World Ba Na Hills tại Đà Nẵng|70000.0  |NULL               |NULL      |NULL        |
|24     |1             |NULL         |1              |Núi Bà Đen 1 ngày                |1000000.0|Tour Di Sản Văn Hóa|NULL      |NULL        |
|10     |2             |NULL         |3              |Lotte World Aquariu

### Cell 6 — Gold Summary Report (chụp screenshot)

In [0]:
# ============================================================
#  GOLD SUMMARY — chụp screenshot cell này cho docs/screenshots/day3/
# ============================================================

GOLD_TABLES = [
    "gold_revenue_daily",
    "gold_provider_performance",
    "gold_tour_performance"
]

print("\n" + "="*55)
print(" GOLD LAYER SUMMARY — DAY 3")
print("="*55)
print(f"{'Table':<30} | {'Records':>8} | {'Status'}")
print("-"*50)

for t in GOLD_TABLES:
    try:
        c = spark.read.table(t).count()
        print(f"{t:<30} | {c:>8,} | --OK--")
    except Exception as e:
        print(f"{t:<30} | {'ERR':>8} | --ERROR-- {str(e)[:30]}")

print("="*55)


 GOLD LAYER SUMMARY — DAY 3
Table                          |  Records | Status
--------------------------------------------------
gold_revenue_daily             |        2 | --OK--
gold_provider_performance      |        2 | --OK--
gold_tour_performance          |        4 | --OK--


### Cell 7 — Kiểm tra kết quả chi tiết

In [0]:
# ============================================================
#  KIỂM TRA CHI TIẾT — để validate logic trước khi gửi [A] Hà
# ============================================================

print("=" * 60)
print("KIỂM TRA 1 — Revenue Daily")
print("=" * 60)
df_rev = spark.read.table("gold_revenue_daily")
print(f"Tổng doanh thu toàn hệ thống: {df_rev.agg(_sum('total_revenue')).collect()[0][0]:,.0f} VNĐ")
print(f"Số ngày có giao dịch: {df_rev.count()}")
df_rev.show(truncate=False)

print("\n" + "=" * 60)
print("KIỂM TRA 2 — Provider Performance")
print("=" * 60)
df_prov = spark.read.table("gold_provider_performance")
print(f"Số providers có doanh thu: {df_prov.count()}")
df_prov.show(truncate=False)

print("\n" + "=" * 60)
print("KIỂM TRA 3 — Tour Performance")
print("=" * 60)
df_tour = spark.read.table("gold_tour_performance")
print(f"Số tours có booking confirmed: {df_tour.count()}")
df_tour.show(truncate=False)

print("\n" + "=" * 60)
print("DATA GỬI CHO [A] HÀ — để validate")
print("=" * 60)
total_rev   = df_rev.agg(_sum("total_revenue")).collect()[0][0] or 0
total_trans = df_rev.agg(_sum("num_transactions")).collect()[0][0] or 0
print(f"  Tổng doanh thu (Gold):    {total_rev:>15,.0f} VNĐ")
print(f"  Tổng số giao dịch:        {total_trans:>15,}")
print(f"  So sánh với silver_revenues count: "
      f"{spark.read.table('silver_revenues').count()}")
print("\n-> Gửi 2 con số trên cho [A] Hà kiểm tra lại với SQL Server gốc")

KIỂM TRA 1 — Revenue Daily
Tổng doanh thu toàn hệ thống: 5,140,000 VNĐ
Số ngày có giao dịch: 2
+------------+-------------+----------------+------------+----------------+-------+
|booking_date|total_revenue|provider_revenue|platform_fee|num_transactions|month  |
+------------+-------------+----------------+------------+----------------+-------+
|NULL        |4000000.0    |3600000.0       |400000.0    |1               |NULL   |
|2026-05-24  |1140000.0    |1026000.0       |114000.0    |3               |2026-05|
+------------+-------------+----------------+------------+----------------+-------+


KIỂM TRA 2 — Provider Performance
Số providers có doanh thu: 2
+----------+----------------------+---------+------------------------+------------------+----------+
|creator_id|total_provider_revenue|total_gmv|total_confirmed_bookings|active_tours_count|avg_rating|
+----------+----------------------+---------+------------------------+------------------+----------+
|11        |900000.0             